In [229]:
import jax
from jax import random
from scipy.stats import *
from scipy import stats 
from typing import Union, List, Literal, TypeAlias
import numpy as np
import numpy.typing as npt
import pandas as pd
from functools import wraps, partial
from arch import arch_model
import time
from dataclasses import dataclass
from jax.scipy.optimize import minimize
import matplotlib.pyplot as plt
from math import exp, pi, sqrt

import jax.numpy as jnp

jax.config.update("jax_enable_x64", True) #for better numerical stability 

number = Union[int, float]
number_like = Union[List[number], number]
array_like = Union[List[number], np.ndarray]
distributions: TypeAlias = Literal["chauchy", "chi2", "expon", "exponpow", "gamma", "lognorm", "norm", "powerlaw", "rayleigh",
                            "uniform", "t", "gumbel_r", "f"]  
FloatArray = npt.NDArray[np.float64]
Floats = Union[float, FloatArray]
Int = Union[int, np.int16, np.int32, np.int64, jnp.int64, jnp.int32, jnp.int16]

from Vares import Auxiliary
import Vares

#1. потестить на несколько дней вперёд 
#2. разные распределения

In [ ]:
@dataclass
class GBMParams:
    volatility: Floats 
    mean: Floats 

def make_gbm_simulator(
        params: GBMParams,
        T: int, 
        granularity: int = 1000
    ): 
    '''Creates a Geometric Brownian Motion simulator with predetermined parameters. 
    Returns a batch of simulated data. The simulated data is distributed lognormally shifted by -1 term.
    Parameters correspond to ones drown from normal with mean (params.mean - (params.volatility**2) / 2)) and std params.volatility'''
    def simulate(n_paths: int, seed = None):
        if seed is not None:
            rng = np.random.default_rng(seed=seed)
        else:
            rng = np.random.default_rng()

        time_grid = T * granularity
        dt = 1 / granularity
        norm = rng.normal(size=(n_paths, time_grid))

        #Simulate log returns paths using GBM.
        d_log_S = (
            (params.mean - ((params.volatility**2) / 2)) * dt 
            + params.volatility * norm * np.sqrt(dt)
        )
        d_log_S = np.insert(d_log_S, 0, np.zeros(n_paths), axis=1)

        cum_log_returns = np.cumsum(d_log_S, axis=-1)
        cum_returns = np.exp(cum_log_returns) - 1 

        return cum_returns
    return simulate 

def _terminal_returns(simulator: callable, n_paths: int, seed = None): 
    '''Helper method for stripping and returning the last column of the return matrix.'''
    cummulative_returns = simulator(n_paths, seed)
    terminal_returns = cummulative_returns[:, -1] #in case of GBM terminal return will be distributed lognormally
    return terminal_returns

In [ ]:
#same as make_gbm_simulator_2 but with array slicing is needed here.
@Auxiliary.timer
def make_gbm_simulator_varying_vol(
        params: GBMParams, 
        T: int, 
        granularity: int = 1000 
): 
    '''Same as make_gbm_simulator(), but allows for varying daily volatility of the underlying. A more flexible solution'''
    if T < params.volatility.shape[0]: 
        raise ValueError('Quantity of forecasted volatility can not exceed number of days to simulte')
    
    elif T > params.volatility.shape[0]:
        print('Number of simulated days exceed number of forecasted volality. Simulation will assume constant long-term volatility.')
    
    def simulate(n_paths, seed = None): 
        if seed is not None:
            rng = np.random.default_rng(seed=seed)
        else:
            rng = np.random.default_rng()

        n_vol = params.volatility.shape[0]

        dt = 1 / granularity
        time_grid = T * granularity
        norm = rng.normal(size=(n_paths, time_grid))

        #version 2
        # d_log_S = np.zeros((n_paths, ))
        for day in range(0, n_vol): 

            if day == 0: 
                q = (day+1) * granularity
                z_values = norm[:, :q]

                _ = (
                (params.mean - ((params.volatility[day]**2) / 2)) * dt 
                + params.volatility[day] * z_values * np.sqrt(dt)
                )
                d_log_S = np.concatenate((d_log_S, _), axis=1) if day != 0 else _
            if  0 < day < params.volatility.shape[0]:
                q = (day+1) * granularity
                p = day * granularity
                z_values = norm[:, p:q]

                _ = (
                (params.mean - ((params.volatility[day]**2) / 2)) * dt 
                + params.volatility[day] * z_values * np.sqrt(dt)
                )
                d_log_S = np.concatenate((d_log_S, _), axis=1) if day != 0 else _

        p = n_vol * granularity
        _ = (
        (params.mean - ((params.volatility[-1]**2) / 2)) * dt 
        + params.volatility[-1] * norm[:, p:] * np.sqrt(dt)
        )
        d_log_S = np.concatenate((d_log_S, _), axis=1)
        # d_log_S = np.insert(d_log_S, 0, np.zeros(n_paths), axis=1)

        cum_log_returns = np.cumsum(d_log_S, axis=-1)
        cum_returns = np.exp(cum_log_returns) - 1 

        return cum_returns
    return simulate


In [717]:
class ProtoPortfolio: 
    def __init__(self, returns, **kwargs): 
        self.returns = returns #(portfolio) returns
        try:
            self.number_of_assets = self.returns.shape[1] 
        except IndexError: 
            self.number_of_assets = 1

        self.kwargs = kwargs.copy()

    def calibrate(self, horizon=1, **kwargs): 
        '''Private method implies variance and mean of returns of assets' returns using GARCH(p, q)
        
        Now it is unable to handle returns across multiple(>1) assets. Also can not handle horizon > 1'''
        am = arch_model(self.returns, vol='Garch', dist='normal', **kwargs)
        result = am.fit(disp='off')
        forecast = result.forecast(horizon=horizon)
        mu = result.params.get("mu", 0)
        sigma2 = forecast.variance.values[0]
        sigma = sigma2 ** (1/2) #garch model estimates the variance of the underlyinh sample
        parameters = GBMParams(volatility=sigma, mean=mu)

        self.parameters = parameters #for testing 

        return None
    
    def _simulate(self, T: int = 10, n_paths: int = 500, granularity: int = 1000):
        '''Simualtes GBM n_paths times. Outputs simulated array of terminal returns.'''
        if self.parameters.volatility.shape[0] == 1: 
            simulator = make_gbm_simulator(self.parameters, T, granularity)
            # simulator = make_gbm_simulator(GBMParams(mean=0.0, volatility=0.2), T, granularity)
        else: 
            simulator = make_gbm_simulator_varying_vol(self.parameters, T, granularity)
        simulated_returns = _terminal_returns(simulator, n_paths)

        return simulated_returns 
    
    def historical_var(self, alpha=0.01, T: int = 10, **kwargs): 
        _ = self._simulate(T, **kwargs)
        return -np.quantile(_, alpha)




In [419]:
#forecast for multiple days ahead
@Auxiliary.timer
def make_gbm_simulator_2(
        params: GBMParams, 
        T: int, 
        granularity: int = 1000 
): 
    if T < params.volatility.shape[0]: 
        raise ValueError('Quantity of forecasted volatility can not exceed number of days to simulte')
    
    elif T > params.volatility.shape[0]:
        print('Number of simulated days exceed number of forecasted volality. Simulation will assume constant long-term volatility.')
    
    def simulate(n_paths, seed = None): 
        if seed is not None:
            rng = np.random.default_rng(seed=seed)
        else:
            rng = np.random.default_rng()

        dt = 1 / granularity

        #version 1
        d_log_S = np.zeros((n_paths, ))
        for day in range(0, T): 
            if day < params.volatility.shape[0]:
                norm = rng.normal(size=(n_paths, granularity))

                _ = (
                (params.mean - ((params.volatility[day]**2) / 2)) * dt 
                + params.volatility[day] * norm * np.sqrt(dt)
                )
                d_log_S = np.concatenate((d_log_S, _), axis=1) if day != 0 else _
            else: 
                norm = rng.normal(size=(n_paths, granularity))
                _ = (
                (params.mean - ((params.volatility[-1]**2) / 2)) * dt 
                + params.volatility[-1] * norm * np.sqrt(dt)
                )
                d_log_S = np.concatenate((d_log_S, _), axis=1)
        # d_log_S = np.insert(d_log_S, 0, np.zeros(n_paths), axis=1)

        cum_log_returns = np.cumsum(d_log_S, axis=-1)
        cum_returns = np.exp(cum_log_returns) - 1 

        return cum_returns
    return simulate

            

        
        

In [420]:
#same as make_gbm_simulator_2 but with array slicing is needed here.
@Auxiliary.timer
def make_gbm_simulator_3(
        params: GBMParams, 
        T: int, 
        granularity: int = 1000 
): 
    if T < params.volatility.shape[0]: 
        raise ValueError('Quantity of forecasted volatility can not exceed number of days to simulte')
    
    elif T > params.volatility.shape[0]:
        print('Number of simulated days exceed number of forecasted volality. Simulation will assume constant long-term volatility.')
    
    def simulate(n_paths, seed = None): 
        if seed is not None:
            rng = np.random.default_rng(seed=seed)
        else:
            rng = np.random.default_rng()

        n_vol = params.volatility.shape[0]

        dt = 1 / granularity
        time_grid = T * granularity
        norm = rng.normal(size=(n_paths, time_grid))

        #version 2
        # d_log_S = np.zeros((n_paths, ))
        for day in range(0, n_vol): 

            if day == 0: 
                q = (day+1) * granularity
                z_values = norm[:, :q]

                _ = (
                (params.mean - ((params.volatility[day]**2) / 2)) * dt 
                + params.volatility[day] * z_values * np.sqrt(dt)
                )
                d_log_S = np.concatenate((d_log_S, _), axis=1) if day != 0 else _
            if  0 < day < params.volatility.shape[0]:
                q = (day+1) * granularity
                p = day * granularity
                z_values = norm[:, p:q]

                _ = (
                (params.mean - ((params.volatility[day]**2) / 2)) * dt 
                + params.volatility[day] * z_values * np.sqrt(dt)
                )
                d_log_S = np.concatenate((d_log_S, _), axis=1) if day != 0 else _

        p = n_vol * granularity
        _ = (
        (params.mean - ((params.volatility[-1]**2) / 2)) * dt 
        + params.volatility[-1] * norm[:, p:] * np.sqrt(dt)
        )
        d_log_S = np.concatenate((d_log_S, _), axis=1)
        # d_log_S = np.insert(d_log_S, 0, np.zeros(n_paths), axis=1)

        cum_log_returns = np.cumsum(d_log_S, axis=-1)
        cum_returns = np.exp(cum_log_returns) - 1 

        return cum_returns
    return simulate


In [371]:
normal_sample = stats.norm.rvs(scale=0.2, size=5000)
p = ProtoPortfolio(normal_sample)
p.calibrate(horizon=10, rescale=False)
print(p.parameters.volatility)
params = GBMParams(mean=0, volatility=0.2)

sim = make_gbm_simulator(params, 11)
sim1 = make_gbm_simulator_2(p.parameters, 11)
sim2 = make_gbm_simulator_3(p.parameters, 11)

returns = []
for simulator in [sim, sim1, sim2]:
    returns.append(_terminal_returns(simulator=simulator, n_paths=200))
    
returns = np.array(returns)


a = returns[0]
b = returns[1]
c = returns[2]

print(np.mean(a), np.mean(b), np.mean(c))

[0.19808283 0.19813198 0.19818019 0.19822748 0.19827387 0.19831936
 0.19836399 0.19840776 0.1984507  0.19849282]
Number of simulated days exceed number of forecasted volality. Simulation will assume constant long-term volatility.
make_gbm_simulator_2() took 0.000008s
Number of simulated days exceed number of forecasted volality. Simulation will assume constant long-term volatility.
make_gbm_simulator_3() took 0.000007s
0.04828840951075874 0.07154803455052468 -0.15289317438209202


In [705]:
# lognormal quantile at alpha=0.01 for underlying normal(mean=0, std=0.2)
alpha = 0.01
mu = -0.00132 - ((0.1998**2)/2)
sigma = 0.1998

# SciPy lognorm parameterization: s = sigma, scale = exp(mu)
value_at_alpha = stats.lognorm.ppf(alpha, s=sigma, scale=np.exp(mu), loc=-1)
print(value_at_alpha)


normal_sample = stats.norm.rvs(scale=0.2, size=5000)
# p = ProtoPortfolio(normal_sample)
# p.calibrate(rescale=False)
# print(p.parameters)

# sim = make_gbm_simulator_3(p.parameters, 1)
# rets = _terminal_returns(sim, 50000)
# _ = Vares.historical_var(rets)
# print(_)

params = GBMParams(mean=0, volatility=0.2)
sim = make_gbm_simulator(params, 1)
rets = _terminal_returns(sim, 50000)
_ = Vares.historical_var(rets)
print(_)

# p = ProtoPortfolio(normal_sample)
# p.calibrate(rescale=False)
# print(p.parameters)
# print(p.historical_var(T=1))


-0.38497005741662504
0.383507514587694


In [ ]:
vars = []
params = []
for _ in range(500): 
    normal_sample = stats.norm.rvs(scale=0.2, size=5000)
    p = ProtoPortfolio(normal_sample)
    p.calibrate(rescale=False)
    params.append(p.parameters.volatility)
    # vars.append(p.historical_var(T=1, n_paths=500))
    vars.append(p.historical_var(T=1, n_paths=5000))

print(np.mean(vars))

0.38454222779741976


In [721]:
Vares.ENABLE_TIMING = False
vars = []
params = GBMParams(mean=0, volatility=0.2)
for _ in range(500): 
    sim = make_gbm_simulator(params, 1)
    # rets = _terminal_returns(sim, 500)
    rets = _terminal_returns(sim, 5000)
    _ = Vares.historical_var(rets)
    vars.append(_)

print(np.mean(vars))

0.38413596174701825


In [709]:
print(np.mean(params))

0.199929177040903
